#  Multi-Class Classification (Physics-Informed Neural Network — PINN)
**Name:** Potla Naga Sai Bharath  
**Gmail:** saibharathpotla29@gmail.com

**Task:** Build a model for classifying the images into lenses using PyTorch or Keras. Pick the most appropriate approach and discuss your strategy.

**Dataset Description:** The dataset consists of three classes strong lensing images with no substructure (`no`), subhalo substructure (`sphere`), and vortex substructure (`vort`). Images are stored as `.npy` files of shape `(1, 150, 150)`, pre-normalized using min-max normalization. Images are resized to `150×150` and converted to RGB for compatibility with the pretrained ResNet50 backbone.

**Evaluation Metrics:** ROC curve (Receiver Operating Characteristic curve) and AUC score (Area Under the ROC Curve), computed per-class and averaged as Macro AUC.
Trained model weights for this task can be found in the respective folder https://drive.google.com/file/d/1m8ZxrEWHWZGVY2L45-r4ylcDh46I_Zvn/view?usp=sharing

Strategy
* Rather than using a plain ResNet50 as a feature extractor, this model embeds astrophysical domain knowledge directly into the network architecture through two physics-inspired components  a **Lensing Kernel Layer** and a **Physics Bottleneck**  making it a true Physics-Informed Neural Network (PINN).
* The **LensingKernelLayer** replaces ResNet50's standard first convolution. Its weights are initialized with radially-symmetric deflection kernels mimicking the gravitational lensing deflection field α(θ) ∝ θ/|θ|², encoding the physical geometry of lensing directly into the first feature extraction stage.
* The pretrained ResNet50 backbone (ImageNet weights) then processes these physics-primed features through its four residual stages (layer1–layer4), producing a 2048-channel feature map.
* The **PhysicsBottleneck** sits on top of the backbone and predicts two physically interpretable latent variables: convergence κ ∈ [0,1] (surface mass density) via a sigmoid head, and shear γ ∈ [−1,1] (tidal distortion) via a tanh head. These are pooled and concatenated with the backbone features into a 2051-dimensional combined vector.
* A three-stage classification head (`BatchNorm1d → Linear → ReLU → Dropout`) progressively reduces 2051 → 256 → 64 → 3 with batch normalisation and dropout at each stage to stabilise training and regularise against overfitting. A final linear layer produces raw logits passed through `softmax` at inference.
* A **physics consistency loss** is applied in Phase 3 training: it penalises violations of the lensing constraint relating the Laplacian of κ to the magnitude of γ, acting as a physics-based regulariser on top of cross-entropy.
* Training follows a **3-phase schedule** over 43 epochs total: Phase 1 (5 epochs) warms up only the classifier head with the backbone frozen; Phase 2 (25 epochs) fine-tunes the full backbone without physics loss; Phase 3 (13 epochs) activates the physics consistency loss at λ=0.1 with a reduced learning rate.
* The optimizer is AdamW with a Cosine Annealing LR scheduler (1e-3 → 1e-4 → 5e-5 across phases, minimum floor 1e-6). Multi-GPU training is supported via `nn.DataParallel` with correct `.module` unwrapping for checkpoint saving.
* Aggressive data augmentation is applied during training: random rotation (±180°), random affine translation (±20%), random scaling (0.8×–1.2×), and random horizontal/vertical flips  reflecting the rotational symmetry of gravitational lensing where no canonical orientation exists. Validation images receive only resizing and tensor conversion.
* The best model checkpoint is tracked and saved based on peak validation Macro AUC, ensuring the final model reflects the best generalisation performance rather than the last epoch.
* Per-class AUC scores are computed via one-vs-rest ROC analysis using `sklearn.metrics.roc_curve`, and a Macro AUC is derived by averaging across all three classes  providing a balanced evaluation metric robust to class imbalance.



Results
The best model checkpoint was reloaded and evaluated on the full validation set:

| Metric | Value |
|---|---|
| Validation Loss | 0.1543 |
| Validation Accuracy | 94.49% |
| Macro AUC | 0.9917 |
| AUC  No Substructure | 0.9918 |
| AUC  Subhalo (Sphere) | 0.9864 |
| AUC   Vortex | 0.9969 |

In [ ]:

# IMPORTS

import os
import gc
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.transforms import functional as TF
from sklearn.metrics import roc_curve, auc, confusion_matrix
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# free any leftover memory
gc.collect()
torch.cuda.empty_cache()
torch.backends.cudnn.benchmark = True

print("Imports done.")

In [ ]:

#  CONFIG

DATA_DIR_TRAIN = "/kaggle/input/datasets/saibharath295/task-7/dataset/train"
DATA_DIR_VAL   = "/kaggle/input/datasets/saibharath295/task-7/dataset/val"
SAVE_DIR       = "./pinn_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

CLASSES      = ["no", "sphere", "vort"]
NUM_CLASSES  = 3
IMAGE_SIZE   = 150
BATCH_SIZE   = 16       # keep small to avoid OOM
NUM_WORKERS  = 2
SEED         = 42

# 3-phase training schedule
WARMUP_EPOCHS   = 5     # phase 1: classifier head only
FINETUNE_EPOCHS = 25    # phase 2: full backbone, no physics loss
PHYSICS_EPOCHS  = 13    # phase 3: full model + physics loss

TOTAL_EPOCHS    = WARMUP_EPOCHS + FINETUNE_EPOCHS + PHYSICS_EPOCHS

LAMBDA_PHYSICS  = 0.1

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE} | GPUs: {NUM_GPUS}")
print(f"Total epochs: {TOTAL_EPOCHS}  "
      f"(warmup={WARMUP_EPOCHS}, finetune={FINETUNE_EPOCHS}, physics={PHYSICS_EPOCHS})")

In [ ]:

#  DATASET + DATALOADERS

class NPYLensingDataset(Dataset):
    def __init__(self, root_dir, classes, transform=None):
        self.classes      = classes
        self.transform    = transform
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.samples      = []

        for cls in classes:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_dir):
                continue
            for fname in os.listdir(cls_dir):
                if fname.endswith(".npy"):
                    self.samples.append((os.path.join(cls_dir, fname),
                                         self.class_to_idx[cls]))
        print(f"Loaded {len(self.samples)} samples from {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = np.squeeze(np.load(path))
        pil = Image.fromarray(np.uint8(img * 255)).convert("RGB")
        if self.transform:
            pil = self.transform(pil)
        return pil, label


class TrainTransform:
    def __init__(self, size=150):
        self.size = size

    def __call__(self, img):
        img = img.resize((self.size, self.size), Image.BILINEAR)
        img = TF.rotate(img, random.uniform(-180, 180))
        img = TF.affine(
            img, angle=0.0,
            translate=(int(random.uniform(-0.2, 0.2) * self.size),
                       int(random.uniform(-0.2, 0.2) * self.size)),
            scale=random.uniform(0.8, 1.2), shear=0.0
        )
        if random.random() < 0.5: img = TF.hflip(img)
        if random.random() < 0.5: img = TF.vflip(img)
        img = TF.to_tensor(img)
        img = TF.normalize(img, [0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])
        return img


class ValTransform:
    def __init__(self, size=150):
        self.size = size

    def __call__(self, img):
        img = img.resize((self.size, self.size), Image.BILINEAR)
        img = TF.to_tensor(img)
        img = TF.normalize(img, [0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])
        return img


train_dataset = NPYLensingDataset(DATA_DIR_TRAIN, CLASSES, TrainTransform(IMAGE_SIZE))
val_dataset   = NPYLensingDataset(DATA_DIR_VAL,   CLASSES, ValTransform(IMAGE_SIZE))

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:

#  MODEL ARCHITECTURE


# --- Physics Lensing Kernel Layer ---
# First conv initialized with radially-symmetric deflection kernels
# mimicking α(θ) ∝ θ/|θ|²
class LensingKernelLayer(nn.Module):
    def __init__(self, out_channels=64, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(3, out_channels, kernel_size=kernel_size,
                              stride=2, padding=kernel_size // 2, bias=False)
        self._init_radial_weights(out_channels, kernel_size)

    def _init_radial_weights(self, out_channels, k):
        with torch.no_grad():
            weights = torch.zeros(out_channels, 3, k, k)
            cx, cy  = k // 2, k // 2
            for f in range(out_channels):
                phase = 2 * np.pi * f / out_channels
                for i in range(k):
                    for j in range(k):
                        dx  = i - cx
                        dy  = j - cy
                        r   = max(np.sqrt(dx**2 + dy**2), 1e-6)
                        val = np.cos(phase)*dx/r**2 + np.sin(phase)*dy/r**2
                        weights[f, :, i, j] = val
            self.conv.weight.copy_(weights)

    def forward(self, x):
        return self.conv(x)


# --- Physics Bottleneck ---
# Predicts κ (convergence) and γ (shear) as interpretable latent variables
class PhysicsBottleneck(nn.Module):
    def __init__(self, in_channels=2048):
        super().__init__()
        self.kappa_head = nn.Sequential(
            nn.Conv2d(in_channels, 256, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 1, kernel_size=1),
            nn.Sigmoid()        # κ ∈ [0,1]
        )
        self.gamma_head = nn.Sequential(
            nn.Conv2d(in_channels, 256, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 2, kernel_size=1),
            nn.Tanh()           # γ ∈ [-1,1]
        )
        self.fusion       = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.feature_pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten())

    def forward(self, feat):
        kappa    = self.kappa_head(feat)            # [B, 1, H, W]
        gamma    = self.gamma_head(feat)            # [B, 2, H, W]
        kappa_v  = self.fusion(kappa)               # [B, 1]
        gamma_v  = self.fusion(gamma)               # [B, 2]
        feat_v   = self.feature_pool(feat)          # [B, 2048]
        combined = torch.cat([feat_v, kappa_v, gamma_v], dim=1)  # [B, 2051]
        return combined, kappa, gamma


# --- Full PINN Model ---
class GravitationalLensingPINN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        self.lensing_conv       = LensingKernelLayer(out_channels=64, kernel_size=7)
        self.bn1                = base.bn1
        self.relu               = base.relu
        self.maxpool            = base.maxpool
        self.layer1             = base.layer1
        self.layer2             = base.layer2
        self.layer3             = base.layer3
        self.layer4             = base.layer4
        self.physics_bottleneck = PhysicsBottleneck(in_channels=2048)

        self.classifier = nn.Sequential(
            nn.BatchNorm1d(2051),
            nn.Linear(2051, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.BatchNorm1d(256),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.BatchNorm1d(64),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.lensing_conv(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        combined, kappa, gamma = self.physics_bottleneck(x)
        return self.classifier(combined), kappa, gamma


# --- Freeze/Unfreeze helpers ---
def freeze_backbone(model):
    m = model.module if isinstance(model, nn.DataParallel) else model
    for layer in [m.layer1, m.layer2, m.layer3, m.layer4]:
        for p in layer.parameters():
            p.requires_grad = False

def unfreeze_backbone(model):
    m = model.module if isinstance(model, nn.DataParallel) else model
    for layer in [m.layer1, m.layer2, m.layer3, m.layer4]:
        for p in layer.parameters():
            p.requires_grad = True

# --- Physics consistency loss (downsampled to prevent OOM + nan) ---
def physics_consistency_loss(kappa, gamma):
    kappa = F.avg_pool2d(kappa, kernel_size=2)
    gamma = F.avg_pool2d(gamma, kernel_size=2)

    kappa_xx = kappa[:, :, 2:, 1:-1] - 2*kappa[:, :, 1:-1, 1:-1] + kappa[:, :, :-2, 1:-1]
    kappa_yy = kappa[:, :, 1:-1, 2:] - 2*kappa[:, :, 1:-1, 1:-1] + kappa[:, :, 1:-1, :-2]
    laplacian_kappa = kappa_xx + kappa_yy

    gamma1    = gamma[:, 0:1, 1:-1, 1:-1]
    gamma2    = gamma[:, 1:2, 1:-1, 1:-1]
    gamma_mag = torch.sqrt(gamma1**2 + gamma2**2 + 1e-8)

    residual = (laplacian_kappa.abs() - gamma_mag).pow(2).mean()

    # guard against nan/inf from physics loss overflow
    if torch.isnan(residual) or torch.isinf(residual):
        return torch.tensor(0.0, device=kappa.device, requires_grad=True)

    return residual


print("Model architecture defined.")

In [ ]:

# TRAINING + VALIDATION FUNCTIONS
scaler = GradScaler("cuda")


def train_one_epoch(model, loader, optimizer, ce_loss_fn,
                    use_physics_loss=False, lambda_physics=0.1):
    model.train()
    running_loss, running_correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, desc="Train", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        with autocast("cuda"):
            logits, kappa, gamma = model(images)
            loss = ce_loss_fn(logits, labels)
            if use_physics_loss:
                phys_loss = physics_consistency_loss(kappa, gamma)
                loss = loss + lambda_physics * phys_loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        preds = torch.argmax(logits, dim=1)
        running_loss    += loss.item() * images.size(0)
        running_correct += (preds == labels).sum().item()
        total           += labels.size(0)

    return running_loss / total, running_correct / total


def validate_one_epoch(model, loader, ce_loss_fn):
    model.eval()
    running_loss, running_correct, total = 0.0, 0, 0
    all_labels, all_probs, all_preds = [], [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Val", leave=False):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with autocast("cuda"):
                logits, _, _ = model(images)
                loss         = ce_loss_fn(logits, labels)

            probs = torch.softmax(logits.float(), dim=1)
            preds = torch.argmax(probs, dim=1)

            running_loss    += loss.item() * images.size(0)
            running_correct += (preds == labels).sum().item()
            total           += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

            del images, labels, logits, probs, preds
            torch.cuda.empty_cache()

    return (running_loss / total,
            running_correct / total,
            np.array(all_labels),
            np.array(all_probs),
            np.array(all_preds))


def compute_macro_auc(y_true, y_probs, num_classes):
    y_bin      = np.eye(num_classes)[y_true]
    auc_scores = {}
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
        auc_scores[i] = auc(fpr, tpr)
    return auc_scores, float(np.mean(list(auc_scores.values())))


print("Training functions defined.")

In [ ]:
# TRAINING LOOP (fresh from epoch 0, stops at 43)

model = GravitationalLensingPINN(num_classes=NUM_CLASSES)
if NUM_GPUS > 1:
    print(f"Using DataParallel on {NUM_GPUS} GPUs")
    model = nn.DataParallel(model)
model = model.to(DEVICE)

ce_loss_fn = nn.CrossEntropyLoss()
optimizer  = optim.AdamW(model.parameters(), lr=1e-3)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6
)

best_val_auc   = -1.0
best_model_wts = copy.deepcopy(
    model.module.state_dict() if isinstance(model, nn.DataParallel)
    else model.state_dict()
)

train_loss_hist, val_loss_hist = [], []
train_acc_hist,  val_acc_hist  = [], []
val_auc_hist                   = []

for epoch in range(TOTAL_EPOCHS):

    # ── Phase control ────────────────────────────────────────────────────────
    if epoch == 0:
        print("\nWarm-up (classifier head only)")
        freeze_backbone(model)
        for g in optimizer.param_groups: g["lr"] = 1e-3

    elif epoch == WARMUP_EPOCHS:
        print("\Fine-tune (full backbone, no physics loss)")
        unfreeze_backbone(model)
        for g in optimizer.param_groups: g["lr"] = 1e-4

    elif epoch == WARMUP_EPOCHS + FINETUNE_EPOCHS:
        print("\Physics-guided (full model + physics loss)")
        for g in optimizer.param_groups: g["lr"] = 5e-5

    use_physics = epoch >= (WARMUP_EPOCHS + FINETUNE_EPOCHS)

    # ── Train ─────────────────────────────────────────────────────────────────
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, ce_loss_fn,
        use_physics_loss=use_physics, lambda_physics=LAMBDA_PHYSICS
    )

    # ── Validate ──────────────────────────────────────────────────────────────
    val_loss, val_acc, y_true, y_probs, y_pred = validate_one_epoch(
        model, val_loader, ce_loss_fn
    )
    auc_scores, macro_auc = compute_macro_auc(y_true, y_probs, NUM_CLASSES)

    scheduler.step()

    train_loss_hist.append(train_loss)
    val_loss_hist.append(val_loss)
    train_acc_hist.append(train_acc)
    val_acc_hist.append(val_acc)
    val_auc_hist.append(macro_auc)

    print(f"Epoch [{epoch+1}/{TOTAL_EPOCHS}]  "
          f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}  ||  "
          f"Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | "
          f"AUC: {macro_auc:.4f}  LR: {optimizer.param_groups[0]['lr']:.2e}")

    if macro_auc > best_val_auc:
        best_val_auc   = macro_auc
        best_model_wts = copy.deepcopy(
            model.module.state_dict() if isinstance(model, nn.DataParallel)
            else model.state_dict()
        )
        torch.save({
            "epoch":                epoch + 1,
            "model_state_dict":     best_model_wts,
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_auc":         best_val_auc,
        }, os.path.join(SAVE_DIR, "best_pinn_checkpoint.pth"))
        print(f"  ✓ Best model saved  (AUC = {best_val_auc:.4f})")

    # clear memory every epoch
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nTraining complete. Best AUC: {best_val_auc:.4f}")

In [ ]:
#LOAD BEST CHECKPOINT + EVALUATE

import gc
gc.collect()
torch.cuda.empty_cache()

# rebuild model fresh
model = GravitationalLensingPINN(num_classes=NUM_CLASSES)
model = model.to(DEVICE)

# load best checkpoint
ckpt  = torch.load(os.path.join(SAVE_DIR, "best_pinn_checkpoint.pth"),
                   map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# freeze all params — no gradients needed
for p in model.parameters():
    p.requires_grad = False

print(f"Loaded checkpoint — epoch {ckpt['epoch']} | AUC {ckpt['best_val_auc']:.4f}")

# evaluation with small batch size to avoid OOM
eval_loader = DataLoader(val_dataset, batch_size=16, shuffle=False,
                         num_workers=2, pin_memory=False)

ce_loss_fn = nn.CrossEntropyLoss()
all_labels, all_probs, all_preds = [], [], []
running_loss, running_correct, total = 0.0, 0, 0

with torch.no_grad():
    for images, labels in tqdm(eval_loader, desc="Evaluating"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        with autocast("cuda"):
            logits, _, _ = model(images)
            loss         = ce_loss_fn(logits, labels)

        probs = torch.softmax(logits.float(), dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss    += loss.item() * images.size(0)
        running_correct += (preds == labels).sum().item()
        total           += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

        del images, labels, logits, probs, preds
        torch.cuda.empty_cache()

y_true  = np.array(all_labels)
y_probs = np.array(all_probs)
y_pred  = np.array(all_preds)

val_loss = running_loss / total
val_acc  = running_correct / total

y_bin      = np.eye(NUM_CLASSES)[y_true]
auc_scores = {}
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
    auc_scores[i] = auc(fpr, tpr)
macro_auc = float(np.mean(list(auc_scores.values())))

# move model off GPU before plotting
model.cpu()
gc.collect()
torch.cuda.empty_cache()

print(f"\n===== FINAL RESULTS =====")
print(f"Validation Loss     : {val_loss:.4f}")
print(f"Validation Accuracy : {val_acc:.4f}  ({val_acc*100:.2f}%)")
print(f"Macro AUC           : {macro_auc:.4f}")
for i, cls in enumerate(CLASSES):
    print(f"  AUC ({cls:>6}) = {auc_scores[i]:.4f}")

In [ ]:

# ROC CURVES

plt.figure(figsize=(8, 6))
colors = ["#2196F3", "#4CAF50", "#F44336"]

for i, cls in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
    plt.plot(fpr, tpr, linewidth=2, color=colors[i],
             label=f"{cls} (AUC = {auc_scores[i]:.4f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title(f"PINN — Multi-Class ROC Curve\nMacro AUC = {macro_auc:.4f}", fontsize=13)
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "roc_curve_pinn.png"), dpi=150)
plt.show()
print("ROC curve saved.")

In [ ]:

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
plt.title(f"PINN — Confusion Matrix\nAcc = {val_acc*100:.2f}%", fontsize=13)
plt.colorbar()

tick_marks = np.arange(NUM_CLASSES)
plt.xticks(tick_marks, CLASSES, fontsize=11)
plt.yticks(tick_marks, CLASSES, fontsize=11)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], "d"),
                 ha="center", va="center", fontsize=13,
                 color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.ylabel("True Label", fontsize=12)
plt.xlabel("Predicted Label", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix_pinn.png"), dpi=150)
plt.show()
print("Raw confusion matrix:")
print(cm)

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(train_loss_hist, label="Train Loss")
axes[0].plot(val_loss_hist,   label="Val Loss")
axes[0].axvline(WARMUP_EPOCHS,                   color="gray", linestyle="--", label="Phase 2")
axes[0].axvline(WARMUP_EPOCHS + FINETUNE_EPOCHS, color="red",  linestyle="--", label="Phase 3")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(train_acc_hist, label="Train Acc")
axes[1].plot(val_acc_hist,   label="Val Acc")
axes[1].axvline(WARMUP_EPOCHS,                   color="gray", linestyle="--")
axes[1].axvline(WARMUP_EPOCHS + FINETUNE_EPOCHS, color="red",  linestyle="--")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True)

axes[2].plot(val_auc_hist, label="Val Macro AUC", color="green")
axes[2].axvline(WARMUP_EPOCHS,                   color="gray", linestyle="--", label="Phase 2")
axes[2].axvline(WARMUP_EPOCHS + FINETUNE_EPOCHS, color="red",  linestyle="--", label="Phase 3")
axes[2].set_title("Validation Macro AUC")
axes[2].set_xlabel("Epoch")
axes[2].legend()
axes[2].grid(True)

plt.suptitle("PINN Training History", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "training_curves_pinn.png"), dpi=150)
plt.show()

In [ ]:

torch.save(
    ckpt["model_state_dict"],
    os.path.join(SAVE_DIR, "final_pinn_model.pth")
)
print("Final model weights saved to:", os.path.join(SAVE_DIR, "final_pinn_model.pth"))